**Part 1: Kaggle GPU Host Setup (Dual Tesla T4)**

Run these cells inside a fresh Kaggle notebook with GPU T4 x2 enabled.

Step 1: Install Ollama and Cloudflared
Downloads and unpacks the Ollama runtime and Cloudflare's tunnel binary into the system execution path.

In [ ]:
%%bash
# Download and install Ollama binary
curl -fsSL https://ollama.com/install.sh | sh

# Download Cloudflare tunnel daemon
curl -L --output cloudflared.deb https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
dpkg -i cloudflared.deb
rm cloudflared.deb

Step 2: Launch Ollama Daemon
Starts Ollama with unrestricted cross-origin requests (OLLAMA_ORIGINS="*") across both visible CUDA devices (0,1) to prevent 403 blocks from tunneled requests.

In [ ]:
import os
import subprocess
import time
import urllib.request

# Terminate any leftover instances
subprocess.run(["pkill", "-9", "-f", "ollama"], stderr=subprocess.DEVNULL)
time.sleep(2)

env = os.environ.copy()
env["PATH"] = f"/usr/local/bin:/usr/bin:{env.get('PATH', '')}"
env["OLLAMA_HOST"] = "0.0.0.0:11434"
env["OLLAMA_ORIGINS"] = "*"
env["CUDA_VISIBLE_DEVICES"] = "0,1"
env["OLLAMA_NUM_PARALLEL"] = "1"

# Launch daemon
log_file = open("/tmp/ollama.log", "w")
ollama_proc = subprocess.Popen(
    ["/usr/bin/ollama", "serve"],
    env=env,
    stdout=log_file,
    stderr=subprocess.STDOUT
)

print(f"Starting Ollama daemon (PID {ollama_proc.pid})...")

# Wait until HTTP server responds
ready = False
for _ in range(20):
    try:
        with urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=2) as resp:
            if resp.status == 200:
                ready = True
                break
    except Exception:
        time.sleep(1)

if not ready:
    with open("/tmp/ollama.log", "r") as f:
        print(f.read())
    raise RuntimeError("Ollama daemon failed to initialize.")

print("Ollama daemon is live and listening on 0.0.0.0:11434.")

Step 3: Pull and Build the Memory-Optimized Model
Pulls qwen2.5:32b and creates a customized entry (qwen2.5:32b-fast) constrained to a 4,096 context window. This prevents the KV cache from spilling over into Kaggle's system RAM.

In [ ]:
import json
import subprocess
import urllib.request
from tqdm import tqdm

model_name = "qwen2.5:32b"
url = "http://127.0.0.1:11434/api/pull"
payload = json.dumps({"name": model_name, "stream": True}).encode("utf-8")
req = urllib.request.Request(
    url, data=payload, headers={"Content-Type": "application/json"}
)

pbar = None
total_bytes = 0

print(f"Pulling {model_name}...")
with urllib.request.urlopen(req) as resp:
  for raw_line in resp:
    if not raw_line:
      continue
    data = json.loads(raw_line.decode("utf-8"))
    status = data.get("status", "")

    # Clean in-place progress bar
    if "total" in data and "completed" in data:
      if pbar is None or data["total"] != total_bytes:
        if pbar:
          pbar.close()
        total_bytes = data["total"]
        pbar = tqdm(
            total=total_bytes,
            unit="B",
            unit_scale=True,
            unit_divisor=1024,
            desc=status,
        )
      pbar.n = data["completed"]
      pbar.refresh()
    else:
      if pbar:
        pbar.set_description(status)

if pbar:
  pbar.close()
print(f"\n{model_name} downloaded successfully!")

# Write custom Modelfile with limited context for VRAM fit
modelfile_content = """FROM qwen2.5:32b
PARAMETER num_ctx 4096
PARAMETER num_gpu 999
"""

with open("Modelfile", "w") as f:
  f.write(modelfile_content)

# Build fast custom model
print("Building qwen2.5:32b-fast...")
res = subprocess.run(
    ["ollama", "create", "qwen2.5:32b-fast", "-f", "Modelfile"],
    capture_output=True,
    text=True,
)
print(res.stdout)
print("Model ready!")

Step 4: Expose Model via Cloudflare Tunnel
Exposes the local Ollama port 11434 to a public trycloudflare.com URL.

In [ ]:
import re
import subprocess
import time

subprocess.run(["pkill", "-9", "-f", "cloudflared"], stderr=subprocess.DEVNULL)
time.sleep(1)

tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:11434"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

public_url = None
print("Starting Cloudflare tunnel...")

for line in iter(tunnel_proc.stdout.readline, ''):
    if "trycloudflare.com" in line:
        match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
        if match:
            public_url = match.group(0)
            print("\n" + "=" * 65)
            print(f"ACTIVE ENDPOINT : {public_url}")
            print(f"OPENAI BASE URL : {public_url}/v1")
            print("=" * 65 + "\n")
            break

**Part 2: Local PC Client Configuration**

Step 5:  Test Streaming via Local Terminal
On your local terminal, send a request to verify the tunnel. Always pass "stream": true to avoid Cloudflare's 100-second idle gateway timeout.

In [ ]:
curl -N https://<YOUR-SUBDOMAIN>.trycloudflare.com/v1/chat/completions \
  -H "Content-Type: application/json" \
  -H "Authorization: Bearer ollama" \
  -d '{
    "model": "qwen2.5:32b-fast",
    "messages": [
      {"role": "user", "content": "Explain tensor parallelism in two short sentences."}
    ],
    "stream": true,
    "temperature": 0.7
  }'

Step 6: Connect Cline (in VS Code)
In VS Code, install the Cline extension.


In [ ]:
Configure these exact settings:

API Provider: OpenAI Compatible

Base URL: https://<YOUR-SUBDOMAIN>[.trycloudflare.com/v1](https://.trycloudflare.com/v1)

API Key: ollama

Model ID: qwen2.5:32b-fast

Click Done.

**Step 7: Give the Agent Live Web Search Capability (MCP)**

To prevent the model from scraping raw news HTML that returns HTTP 403 bot blocks, configure official search tools inside Cline's Model Context Protocol (MCP) settings.

In [ ]:
curl -LsSf https://astral.sh/uv/install.sh | sh
source ~/.bashrc

In [ ]:
{
  "mcpServers": {
    "fetch": {
      "command": "uvx",
      "args": ["mcp-server-fetch"]
    },
    "duckduckgo-search": {
      "command": "uvx",
      "args": ["duckduckgo-mcp", "serve"]
    }
  }
}